# 🧠 Proje #17: Gelişmiş Arka Plan Silici (RemBG & U2-Net Derin Derinlemesine İnceleme)

Bu notebook dosyasında, görüntülerden arka plan temizleme işleminde kullanılan modern yapay zeka teknolojilerini, derin öğrenme mimarilerini ve koda dökülen kütüphane altyapılarını detaylıca ele alacağız. Projemiz temel düzeyde bir masaüstü uygulaması olsa da, arka planda oldukça karmaşık bilgisayarlı görü (computer vision) mekanizmaları çalışmaktadır.

## 1. Arka Plan Silme Problemi: Salient Object Detection (Göze Çarpan Nesne Tespiti) vs. Image Matting

Görüntülerden arka planı ayırma işlemi bilgisayarlı görü dünyasında iki ana başlık altında incelenir:

1. **Salient Object Detection (SOD):** Görüntüdeki en belirgin, ilgi çeken nesnenin (insan, hayvan, nesne vb.) sınırlarını belirleyerek arka plandan kabaca ayrıştırılması işlemidir. Burada çıktı genellikle ikili bir maskedir ($0$ veya $1$).
2. **Image Matting:** Çok daha hassas bir işlemdir. Saç telleri, cam yansımaları, tüy ve ağaç yaprakları gibi yarı saydam ve son derece ince yapıların arka plandan piksel düzeyinde ayrıştırılmasını hedefler. Görüntüyü şu formülle ifade eder:
   $$I = \alpha F + (1 - \alpha) B$$
   * $I$: Gözlemlenen orijinal görüntü
   * $F$: Ön plan (Foreground)
   * $B$: Arka plan (Background)
   * $\alpha$: Saydamlık katsayısı ($0$ ile $1$ arasında değer alır). $\alpha = 0$ ise piksel tamamen arka plana, $\alpha = 1$ ise tamamen ön plana aittir.

Geleneksel yöntemler (örneğin **GrabCut** algoritması) renk histogramları ve kenar geçişlerini kullanırken, karmaşık arka planlarda ve benzer renk tonlarında tamamen çuvallamaktadır. Bu yüzden günümüzde derin öğrenme tabanlı semantik segmentasyon modelleri tercih edilmektedir.

## 2. U2-Net Yapay Zeka Mimarisi

Projemizde kullandığımız `rembg` kütüphanesi varsayılan olarak **U2-Net** (U-square Net) derin öğrenme mimarisini kullanır.

### U2-Net Neden Farklıdır?
Geleneksel segmentasyon modelleri (UNet gibi) yüksek çözünürlüklü görüntü özelliklerini korumak için ağır omurgalara (backbone: ResNet, VGG vb.) ihtiyaç duyar. Bu da hem dosya boyutunu büyütür hem de işlem hızını yavaşlatır. U2-Net ise **iki katmanlı U yapısı** (Nested U-Structure) sunar:

1. **RSU (Residual U-Blocks):** U2-Net'in temel yapı taşıdır. RSU blokları, kendi içinde küçük U-Net benzeri encoder-decoder yapıları barındırır. Bu sayede, havuzlama (pooling) işlemlerinden kaynaklanan bilgi kaybını azaltarak farklı ölçeklerdeki özellikleri (multi-scale features) doğrudan blok seviyesinde yakalar.
2. **Hafif ve Güçlü Mimari:** RSU blokları sayesinde derin ve ağır harici omurgalar kullanmadan zengin özellik haritaları çıkarılabilir. Saç telleri gibi aşırı detaylı alanlarda bu mimari benzersiz bir hassasiyet sunar.

## 3. ONNX ve ONNX Runtime Nedir?

Modelin çalıştırılması sırasında konsolda `onnxruntime` paketinin gerekliliğiyle karşılaştık. Peki nedir bu ONNX?

- **ONNX (Open Neural Network Exchange):** Farklı yapay zeka kütüphaneleri (PyTorch, TensorFlow, Keras vb.) arasında eğitilmiş modellerin taşınabilir olmasını sağlayan ortak bir formattır. Model PyTorch ile eğitilip, ONNX formatına dönüştürülebilir.
- **ONNX Runtime:** ONNX modellerinin CPU veya GPU üzerinde maksimum performansla çalıştırılmasını (inference) sağlayan yüksek düzeyde optimize edilmiş bir çalışma motorudur (engine). C++ ile yazılmıştır ve donanıma özel hızlandırmalar (AVX, CUDA vb.) kullanarak Python üzerinde PyTorch'a kıyasla çok daha hızlı tahmin (inference) üretir.

## 4. Kullanılan Kütüphanelerin Analizi

### A. `rembg`
Arka plan silme işlemini tek fonksiyona indirgeyen bu kütüphane şu adımları otomatik olarak yürütür:
1. Girdiyi (Image, bytes veya numpy dizisi) alır.
2. Görüntüyü U2-Net modelinin beklediği çözünürlüğe (örn. $320 \times 320$) getirir ve normalize eder.
3. ONNX Runtime üzerinden modeli koşturarak ön plan maskesini (alpha kanalı) üretir.
4. Maskeyi orijinal çözünürlüğe ölçeklendirir, kenarları yumuşatır (alpha matting) ve orijinal görsele maske olarak uygulayıp transparan PNG formatında çıktı verir.

### B. `Pillow (PIL)`
Python'daki standart görsel işleme kütüphanesidir. `rembg` çıktısını alırken veya görseli Tkinter arayüzünde ölçeklendirip gösterirken Pillow'dan yararlanırız.

### C. `Tkinter`
Python'ın built-in grafik arayüz kütüphanesidir. Olay döngüsü (Event Loop - `mainloop()`) mimarisiyle çalışır. Yani pencere kapatılana kadar sürekli olarak kullanıcının tıklama, sürükleme gibi olaylarını dinler.

## 5. Tip Güvenliği Hatasının (`Image | bytes | ndarray`) Teknik Detayı

Projede karşılaştığımız tip uyarısının teknik analizi şöyledir:

```python
cikti_veri = remove(girdi_veri)
o.write(cikti_veri)  # <- Tip denetleyici burada hata verir
```

`rembg.remove` fonksiyonunun dönüş tipi kütüphanede şu şekilde deklare edilmiştir:
```python
def remove(
    data: Union[PIL.Image.Image, bytes, numpy.ndarray],
    ...
) -> Union[PIL.Image.Image, bytes, numpy.ndarray]:
```

Python'daki `open(..., 'wb')` ile açılan dosyanın `write()` metodu ise yalnızca `Buffer` protokolünü destekleyen tipleri kabul eder. Tip denetleyicisi (Pylance), `cikti_veri` değişkeninin bir `Image` veya `ndarray` olabileceğini varsayar. Bu nesneler doğrudan dosyaya yazılamayacağı için hata verir.

**Çözüm (Type Guard):**
```python
if isinstance(cikti_veri, bytes):
    o.write(cikti_veri)
```
`isinstance` kontrolü ekleyerek statik analize bu satıra gelindiğinde verinin **kesinlikle** `bytes` türünde olduğunu ispatlamış oluruz. Bu sayede hem tip uyarısı ortadan kalkar hem de kodumuz daha güvenli çalışır.

## 6. Performans Optimizasyonu: GUI Kitlenmesini Önleme (Threading)

U2-Net gibi derin öğrenme modelleri CPU üzerinde çalışırken yoğun matematiksel hesaplamalar yapar. Bu hesaplama işlemi 1 ila 5 saniye (ilk yüklemede daha uzun) sürebilir.

Eğer bu işlemi doğrudan Tkinter'ın ana çalışma kanalında (Main Thread) yaparsak, işletim sistemi arayüze veri gönderemediği için uygulama donar ve "Yanıt Vermiyor" durumuna düşer.

### Profesyonel Çözüm:
Görüntü işleme adımını ayrı bir kanala (`threading.Thread`) devrederek arayüzün akıcı kalmasını sağlayabiliriz:

```python
import threading

def arka_plan_sil_thread():
    # Ayrı kanalda çalıştır
    t = threading.Thread(target=arka_plan_sil_islemi, daemon=True)
    t.start()
```

Bu sayede model çalışırken arayüzde bir yükleniyor animasyonu/ilerleme çubuğu gösterebilir ve kullanıcının programı kapatmasını veya iptal etmesini sağlayabiliriz.

## 7. Uygulamalı Gösterim (Jupyter Üzerinde Çalıştırma)

Aşağıdaki kod bloğunda, `rembg` fonksiyonunun Jupyter Notebook ortamında doğrudan nasıl çalıştırıldığını ve görsellerin Jupyter üzerinde nasıl görüntülenebileceğini inceleyebilirsiniz.

In [ ]:
from rembg import remove
from PIL import Image
import io

# Eğer projenizde test etmek isterseniz:
# 1. Giriş görselini belirleyin (Örnek olarak kendi public klasörümüzdeki ekran görüntüsünü kullanalım)
giris_yolu = "./public/arka-plan-sil.png"

try:
    # Görseli PIL kullanarak açma
    input_image = Image.open(giris_yolu)
    print("Görsel başarıyla yüklendi. Çözünürlük:", input_image.size)
    
    # Arka planı silme (Burada girdi PIL Image nesnesidir, çıktı da PIL Image olacaktır)
    output_image = remove(input_image)
    
    # Görselleri notebook içinde gösterme
    print("Orijinal Görsel:")
    display(input_image)
    
    print("Arka Planı Silinmiş Görsel (Transparan):")
    display(output_image)
    
except FileNotFoundError:
    print(f"Test edilecek görsel bulunamadı: {giris_yolu}. Lütfen yolu kontrol edin.")
except Exception as e:
    print("Bir hata oluştu:", str(e))